# 타이타닉 생존자 예측

공개 명단의 생존 여부로 분류 모델을 학습하고, 비공개 명단의 생존 여부를 예측합니다. 검증 데이터는 모델 평가에만 사용하고, 마지막에는 공개된 라벨 전체로 다시 학습합니다.

In [18]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [19]:
# input.csv에는 생존 라벨이 있고, titanic_submission.csv에는 예측할 대상이 있습니다.
train_df = pd.read_csv("input.csv", encoding="utf-8")
hidden_df = pd.read_csv("titanic_submission.csv", encoding="utf-8")

X = train_df.drop(columns="Survived")
y = train_df["Survived"]

print(f"학습 데이터: {len(train_df)}명 / 예측 데이터: {len(hidden_df)}명")
print("생존 라벨 분포:")
print(y.value_counts(normalize=True).sort_index().rename("비율").round(3))
display(train_df.head())

학습 데이터: 791명 / 예측 데이터: 100명
생존 라벨 분포:
Survived
0    0.617
1    0.383
Name: 비율, dtype: float64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 특징 만들기와 전처리

이름에서 호칭을 추출하고 가족 규모 및 객실 갑판 정보를 만듭니다. 결측치 대체와 원-핫 인코딩은 파이프라인 안에서 학습 데이터에만 맞춰집니다. 나무 기반 모델은 스케일링이 필요하지 않고, 식별자나 등급·가족 수를 무조건 이상치로 자르는 것도 적절하지 않아 IQR clipping은 적용하지 않습니다.

In [20]:
def make_features(data):
    features = data.copy()

    title = features["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
    title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    features["Title"] = title.where(title.isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")

    features["FamilySize"] = features["SibSp"] + features["Parch"] + 1
    features["IsAlone"] = (features["FamilySize"] == 1).astype(int)
    features["HasCabin"] = features["Cabin"].notna().astype(int)
    features["Deck"] = features["Cabin"].str[0].fillna("Unknown")

    return features.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])


X = make_features(X)
X_hidden = make_features(hidden_df)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_cols),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_cols,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=42,
            ),
        ),
    ]
)

## 검증 및 전체 학습 데이터 예측

먼저 공개 데이터 일부를 떼어 정확도를 확인합니다. 성능 확인 후 같은 전처리·모델 파이프라인을 공개 데이터 전체에 다시 학습하고 비공개 명단을 예측하여 `titanic_predictions.csv`로 저장합니다.

In [21]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
valid_pred = model.predict(X_valid)

print(f"검증 정확도: {accuracy_score(y_valid, valid_pred):.3f}")
print("혼동 행렬 (실제 행, 예측 열):")
print(confusion_matrix(y_valid, valid_pred))
print(classification_report(y_valid, valid_pred, zero_division=0))

# 검증에만 쓰였던 데이터까지 포함해 공개 라벨 전체로 최종 모델을 학습합니다.
model.fit(X, y)
hidden_pred = model.predict(X_hidden)

submission = pd.DataFrame(
    {
        "PassengerId": hidden_df["PassengerId"],
        "Survived": hidden_pred,
    }
)
assert len(submission) == len(hidden_df)
assert submission["PassengerId"].equals(hidden_df["PassengerId"])
assert submission["Survived"].isin([0, 1]).all()

submission.to_csv("titanic_predictions.csv", index=False)
print("예측 결과 저장 완료: titanic_predictions.csv")
display(submission.head())
print("예측 생존 여부 분포:")
print(submission["Survived"].value_counts().sort_index())

검증 정확도: 0.824
혼동 행렬 (실제 행, 예측 열):
[[82 16]
 [12 49]]
              precision    recall  f1-score   support

           0       0.87      0.84      0.85        98
           1       0.75      0.80      0.78        61

    accuracy                           0.82       159
   macro avg       0.81      0.82      0.82       159
weighted avg       0.83      0.82      0.82       159

예측 결과 저장 완료: titanic_predictions.csv


,PassengerId,Survived
0,660,0
1,353,0
2,667,0
3,545,1
4,10,1


예측 생존 여부 분포:
Survived
0    67
1    33
Name: count, dtype: int64
